In [0]:
dbutils.widgets.dropdown(
    "selectdate",
     "2025-09-07",
     ["2025-09-07", "2025-09-06", "2025-09-05"],
     "Select date"
)


In [0]:
stores={
  "2025-09-06": "/Volumes/workspace/default/projectfiles/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-07-2025.json",
  "2025-09-05": "/Volumes/workspace/default/projectfiles/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-06-2025.json",
  "2025-09-07": "/Volumes/workspace/default/projectfiles/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-08-2025.json"
}
selected_date=dbutils.widgets.get("selectdate") 


In [0]:
store_path=stores[selected_date]
print(store_path)

day4_data=spark.read.json(store_path, multiLine=True)
day4_data.display()


In [0]:
# to get valid data
from pyspark.sql.functions import *
day4_valid=day4_data.filter((col("isVoid")==False)|(col("isVoid").isNull()))
day4_valid.display()

In [0]:
# find daily sales payment with payment type aggregation
from pyspark.sql.functions import col, explode, when

# Explode payments.cards
cards_exploded = day4_valid.withColumn("cards",explode(col("payments.cards")))

# Explode card.detail
cards_exploded = cards_exploded.withColumn("details",explode(col("cards.detail")))

# Select relevant fields
cards_exploded = cards_exploded.select(
    col("details.amount").alias("amount"),
    col("details.bankName").alias("bankName"),
    col("details.bankType").alias("bankType"),
    col("details.onlineName").alias("onlineName"),
    col("details.onlineType").alias("onlineType"),
    col("cards.cardType").alias("cardType")
)

# Add paymentMethod column using extracted fields
cards_exploded = cards_exploded.withColumn(
    "paymentMethod",
    when(col("cardType") == "CreditCard", col("bankName"))
    .when(col("cardType") == "Online", col("onlineName")).otherwise(col("cardType"))
)

cash_payments=day4_valid.select(explode(col("payments.cash")).alias("amount")).withColumn("paymentMethod",lit("cash"))
cash_payments.display()

card_payments=cards_exploded.select("paymentMethod","amount")
card_payments.display()

payments=card_payments.unionByName(cash_payments)
payments=payments.groupBy(col("paymentMethod")).agg(sum(col("amount")).alias("total_amount"))

payment_final=payments.withColumn("date",lit(selected_date))\
    .withColumn("deployment_Name",lit("ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 1005004"))

payment_final.display()


In [0]:
# find net sales per store
net_sales=day4_valid.